[![image](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/opengeos/HyperCoast/blob/main/docs/examples/appeears_emit_area.ipynb)

# Request EMIT hyperspectral bands with AppEEARS

This notebook uses a compact HyperCoast API to request selected EMIT surface reflectance bands from NASA AppEEARS for a small area.

In [ ]:
# %pip install "hypercoast[extra]"

In [ ]:
import hypercoast

Log in to AppEEARS. This uses your saved Earthdata credentials, environment variables, or netrc credentials when available.

In [ ]:
client = hypercoast.appeears_login()

Choose a small set of EMIT reflectance wavelengths. HyperCoast converts these wavelengths to the nearest AppEEARS band layers.

In [ ]:
wavelengths = [560, 650, 865, 1640, 2210]
layers = hypercoast.appeears_emit_layers(wavelengths, client=client)
layers

Build and submit a small area request. The bounding box is `[xmin, ymin, xmax, ymax]` in longitude and latitude.

In [ ]:
bbox = [-118.45, 33.85, -118.25, 34.05]

task = hypercoast.appeears_area_task(
    task_name="emit_hyperspectral_area",
    geometry=bbox,
    layers=layers,
    start_date="2024-04-01",
    end_date="2024-04-30",
)

response = hypercoast.appeears_submit_task(task, client=client)
response

Wait for the task, download the GeoTIFF files, and stack them into a small hyperspectral cube.

In [ ]:
task_id = response["task_id"]
hypercoast.appeears_wait(task_id, client=client, interval=60)

files = hypercoast.appeears_download(
    task_id,
    out_dir="data/appeears_emit_area",
    client=client,
    file_types=["tif"],
)
files[:3]

In [ ]:
metadata = hypercoast.appeears_emit_layers(
    wavelengths,
    client=client,
    return_metadata=True,
)

dataset = hypercoast.read_appeears(files, layers=metadata)
dataset

In [ ]:
dataset["reflectance"].sel(wavelength=650, method="nearest").plot.imshow(cmap="viridis")